In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.model_selection import train_test_split 
from sklearn.impute import SimpleImputer 
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

In [3]:
data = pd.read_csv("titanic_data.csv") 
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,no,third,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,yes,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,yes,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,yes,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,no,third,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
data.drop(["PassengerId","Name","Ticket"],axis=1,inplace=True) 

In [6]:
data["FamilySize"] = data["SibSp"] + data["Parch"] + 1

In [7]:
data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,FamilySize
0,no,third,male,22.0,1,0,7.2500,NaN,S,2
1,yes,first,female,38.0,1,0,71.2833,C85,C,2
2,yes,third,female,26.0,0,0,7.9250,NaN,S,1
3,yes,first,female,35.0,1,0,53.1000,C123,S,2
4,no,third,male,35.0,0,0,8.0500,NaN,S,1


In [10]:
x = data.drop("Survived",axis=1)  # Feature Col

In [13]:
y = data["Survived"] # Target Col

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
x, y, test_size=0.2, random_state=42)

In [15]:
## Impute Data

In [16]:
imputation_data = ColumnTransformer(
    transformers=[
        ("age",SimpleImputer(missing_values=np.nan,strategy="mean"),["Age"]), 
        ("embarked",SimpleImputer(missing_values=np.nan,strategy="most_frequent"),["Embarked"]), 
        ("cabin",SimpleImputer(missing_values=np.nan,strategy="constant",fill_value="Missing",add_indicator=True),["Cabin"])
    ], 
    remainder="passthrough", 
    verbose_feature_names_out=False
) 
imputation_data.set_output(transform="pandas") 
imputation_data.fit(X_train) 
X_train = imputation_data.transform(X_train) 
X_test = imputation_data.transform(X_test)

In [17]:
X_train.isnull().sum()

Age                       0
Embarked                  0
Cabin                     0
missingindicator_Cabin    0
Pclass                    0
Sex                       0
SibSp                     0
Parch                     0
Fare                      0
FamilySize                0
dtype: int64

In [18]:
X_test.isnull().sum()

Age                       0
Embarked                  0
Cabin                     0
missingindicator_Cabin    0
Pclass                    0
Sex                       0
SibSp                     0
Parch                     0
Fare                      0
FamilySize                0
dtype: int64

In [19]:
# Outlier Handeling

In [23]:
# Z Score

In [20]:
mean_age = X_train["Age"].mean() 
std_age = X_train["Age"].std() 
X_train["zscore_age"] = (X_train["Age"]-mean_age) / std_age 
X_train = X_train[abs(X_train["zscore_age"]) <= 3] 
X_train.drop(["zscore_age"],axis=1,inplace=True) 

C:\Users\samio\AppData\Local\Temp\ipykernel_2508\1313929508.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train.drop(["zscore_age"],axis=1,inplace=True)


In [22]:
X_train.head()

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,SibSp,Parch,Fare,FamilySize
331,45.5,S,C124,False,first,male,0,0,28.5000,1
733,23.0,S,Missing,True,second,male,0,0,13.0000,1
382,32.0,S,Missing,True,third,male,0,0,7.9250,1
704,26.0,S,Missing,True,third,male,1,0,7.8542,2
813,6.0,S,Missing,True,third,female,4,2,31.2750,7


In [25]:
# IQR Method - Fare

In [26]:
fare_Q1 = X_train["Fare"].quantile(0.25) 
fare_Q3 = X_train["Fare"].quantile(0.75) 
fare_IQR = fare_Q3 - fare_Q1 
fare_min = max(0,fare_Q1 - 1.5*fare_IQR) 
fare_max = fare_Q3 + 1.5*fare_IQR 

X_train["Fare"] = X_train["Fare"].clip(fare_min,fare_max) 

C:\Users\samio\AppData\Local\Temp\ipykernel_2508\3549930339.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train["Fare"] = X_train["Fare"].clip(fare_min,fare_max)


In [28]:
X_train.sample(5)

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,SibSp,Parch,Fare,FamilySize
348,3.0,S,Missing,True,third,male,1,1,15.9000,3
262,52.0,S,E67,False,first,male,1,1,64.3625,3
713,29.0,S,Missing,True,third,male,0,0,9.4833,1
269,35.0,S,C99,False,first,female,0,0,64.3625,1
293,24.0,S,Missing,True,third,female,0,0,8.8500,1


In [29]:
## Encoading and Scaling

In [30]:
X_train["Cabin_Deck"] = X_train["Cabin"].astype(str).str[0] 
X_test["Cabin_Deck"] = X_test["Cabin"].astype(str).str[0]

In [31]:
X_train.sample(5)

,Age,Embarked,Cabin,missingindicator_Cabin,Pclass,Sex,SibSp,Parch,Fare,FamilySize,Cabin_Deck
463,48.000000,S,Missing,True,second,male,0,0,13.0000,1,M
667,29.498846,S,Missing,True,third,male,0,0,7.7750,1,M
400,39.000000,S,Missing,True,third,male,0,0,7.9250,1,M
733,23.000000,S,Missing,True,second,male,0,0,13.0000,1,M
404,20.000000,S,Missing,True,third,female,0,0,8.6625,1,M


In [32]:
encoder_scaler = ColumnTransformer(
    transformers= [
        ("pclass",OrdinalEncoder(categories=[["third","second","first"]]),["Pclass"]), 
        ("embarked_sex_cabin",OneHotEncoder(sparse_output=False,drop="first"),["Embarked","Sex","Cabin_Deck"]), 
        ("age_scaler",StandardScaler(),["Age"]), 
        ("fare_scaler",MinMaxScaler(),["Fare","FamilySize"])
    ] ,remainder="passthrough", 
    verbose_feature_names_out=False
) 
encoder_scaler.set_output(transform="pandas") 
encoder_scaler.fit(X_train) 
X_train = encoder_scaler.transform(X_train) 
X_test = encoder_scaler.transform(X_test) 

In [33]:
X_train

,Pclass,Embarked_Q,Embarked_S,Sex_male,Cabin_Deck_B,Cabin_Deck_C,Cabin_Deck_D,Cabin_Deck_E,Cabin_Deck_F,Cabin_Deck_G,Cabin_Deck_M,Cabin_Deck_T,Age,Fare,FamilySize,Cabin,missingindicator_Cabin,SibSp,Parch
331,2.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.304495,0.442804,0.0,C124,False,0,0
733,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.495295,0.201981,0.0,Missing,True,0,0
382,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.224621,0.123131,0.0,Missing,True,0,0
704,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.255323,0.122031,0.1,Missing,True,1,0
813,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.855135,0.485920,0.6,Missing,True,4,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.655276,0.118858,0.0,Missing,True,0,0
270,2.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.024552,0.481647,0.0,Missing,True,0,0
860,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.944537,0.219201,0.2,Missing,True,2,0
435,2.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.215210,1.000000,0.3,B96 B98,False,1,2


In [34]:
X_train.drop(["Cabin","SibSp","Parch"],axis=1,inplace=True)

In [35]:
X_test.drop(["Cabin","SibSp","Parch"],axis=1,inplace=True)

In [36]:
## Final Dataset

In [37]:
X_train

,Pclass,Embarked_Q,Embarked_S,Sex_male,Cabin_Deck_B,Cabin_Deck_C,Cabin_Deck_D,Cabin_Deck_E,Cabin_Deck_F,Cabin_Deck_G,Cabin_Deck_M,Cabin_Deck_T,Age,Fare,FamilySize,missingindicator_Cabin
331,2.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.304495,0.442804,0.0,False
733,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.495295,0.201981,0.0,True
382,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.224621,0.123131,0.0,True
704,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.255323,0.122031,0.1,True
813,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.855135,0.485920,0.6,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.655276,0.118858,0.0,True
270,2.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.024552,0.481647,0.0,True
860,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.944537,0.219201,0.2,True
435,2.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.215210,1.000000,0.3,False


In [38]:
X_test

,Pclass,Embarked_Q,Embarked_S,Sex_male,Cabin_Deck_B,Cabin_Deck_C,Cabin_Deck_D,Cabin_Deck_E,Cabin_Deck_F,Cabin_Deck_G,Cabin_Deck_M,Cabin_Deck_T,Age,Fare,FamilySize,missingindicator_Cabin
709,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.024552,0.236874,0.2,True
439,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.144630,0.163138,0.0,True
840,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.735267,0.123131,0.0,True
720,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.855135,0.512721,0.1,True
39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.215210,0.174662,0.1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
433,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.975238,0.110701,0.0,True
773,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.024552,0.112255,0.0,True
25,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.704565,0.487668,0.6,True
84,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.975238,0.163138,0.0,True
